> title : 제 4회 ETRI 휴먼이해 인공지능 논문경진대회 <br>
> author : hjy,byc <br>

### 📦 라이브러리

In [1]:
! pip install haversine >/dev/null
! pip install optuna  >/dev/null
! pip install category_encoders >/dev/null
! pip install tabpfn  >/dev/null
! pip install catboost >/dev/null
! pip install torchmetrics >/dev/null

In [2]:
# 기본 모듈
import os
import sys
import re
import ast
import glob
import random
import warnings
from collections import Counter
from math import radians, cos, sin, asin, sqrt
from functools import reduce
from datetime import datetime, timedelta, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 머신러닝
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, roc_curve
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb
from tabpfn import TabPFNClassifier

# PyTorch
import torch
from torch import nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Hugging Face
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    LlamaTokenizer,
    LlamaForCausalLM,
    LlamaForSequenceClassification
)

# PEFT (Parameter-Efficient Fine-Tuning)
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    TaskType
)

# Evaluation & Utilities
from torchmetrics import Accuracy

# 기타
from tqdm import tqdm
from tqdm.auto import tqdm as auto_tqdm  # 필요 시 구분
from scipy.stats import entropy
from haversine import haversine
from io import StringIO
import gc

# wandb
import wandb
wandb.login(key="5fa8dfb2c5be3c888bfe0101437a8fa22fbdf0e0")
wandb.init(project="etri_lifelog", entity="byc3230")

pd.set_option('display.max_columns', 999)
pd.set_option('display.max_rows', 999)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

# 기타
warnings.filterwarnings('ignore')

2025-06-14 03:16:07.357800: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749870967.553459      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749870967.612290      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.

In [3]:
string = """
subject_id,sleep_date
id01,2024-07-24
id01,2024-08-26
id01,2024-08-28
id01,2024-08-29
id02,2024-08-23
id02,2024-09-24
id02,2024-09-26
id02,2024-09-27
id03,2024-08-30
id03,2024-09-01
id03,2024-09-02
id03,2024-09-06
id04,2024-09-03
id04,2024-10-10
id04,2024-10-12
id04,2024-10-13
id05,2024-10-19
id05,2024-10-23
id05,2024-10-24
id05,2024-10-27
id06,2024-07-25
id06,2024-07-26
id06,2024-07-27
id06,2024-07-30
id07,2024-07-07
id07,2024-08-02
id07,2024-08-04
id07,2024-08-05
id08,2024-08-28
id08,2024-08-29
id08,2024-08-30
id08,2024-09-02
id09,2024-08-02
id09,2024-08-31
id09,2024-09-02
id09,2024-09-03
id10,2024-08-28
id10,2024-08-30
id10,2024-08-31
id10,2024-09-03
"""

# DataFrame 생성
valid_ids = pd.read_csv(StringIO(string), sep=',')
valid_ids['pk'] = valid_ids['subject_id']+valid_ids['sleep_date']

### 📦 데이터 읽기

In [4]:
# from google.colab import drive, files
# drive.mount('/content/drive')

# google path
path = '/content/drive/MyDrive/data/ch2025_data_items/share/'

# kaggle path
#path = '../input/'

# [1]공통
train = pd.read_parquet(f'{path}train_63775_v2.parquet')
test = pd.read_parquet(f'{path}test_63775_v2.parquet')

# [2]version2 train,test 데이터셋 by 현종열
# train = pd.read_parquet(f'{path}train_hjy_0603_v1.parquet')
# test = pd.read_parquet(f'{path}test_hjy_0603_v1.parquet')

# [3]QWEN3 8B 활용한 결측처리 (대상: mScreenStatus)
# mScreenStatus_llm = pd.read_excel(f'{path}mScreenStatus_llm결측값생성후파생변수생성_20250609_v1.xlsx')
# feats = ['sleep_time', 'wake_time', 'sleep_duration_min', 'avg_sleep_time', 'avg_wake_time', 'avg_sleep_duration', 'sleep_time_diff', 'wake_time_diff', 'sleep_duration_diff', 'sleep_time_ratio', 'wake_time_ratio', 'sleep_duration_ratio', 'sleep_time_lag1', 'wake_time_lag1', 'sleep_duration_lag1', 'sleep_time_diff_lag1', 'wake_time_diff_lag1', 'sleep_duration_diff_lag1', 'sleep_time_ratio_lag1', 'wake_time_ratio_lag1', 'sleep_duration_ratio_lag1', 'sleep_time_lag2', 'wake_time_lag2', 'sleep_duration_lag2', 'sleep_time_diff_lag2', 'wake_time_diff_lag2', 'sleep_duration_diff_lag2', 'sleep_time_ratio_lag2', 'wake_time_ratio_lag2', 'sleep_duration_ratio_lag2', 'sleep_time_mean2d', 'wake_time_mean2d', 'sleep_duration_min_mean2d', 'sleep_time_diff_mean2d', 'wake_time_diff_mean2d', 'sleep_duration_diff_mean2d', 'sleep_time_ratio_mean2d', 'wake_time_ratio_mean2d', 'sleep_duration_ratio_mean2d', 'sleep_time_std2d', 'wake_time_std2d', 'sleep_duration_min_std2d', 'sleep_time_diff_std2d', 'wake_time_diff_std2d', 'sleep_duration_diff_std2d', 'sleep_time_ratio_std2d', 'wake_time_ratio_std2d', 'sleep_duration_ratio_std2d', 'sleep_time_mean3d', 'wake_time_mean3d', 'sleep_duration_min_mean3d', 'sleep_time_diff_mean3d', 'wake_time_diff_mean3d', 'sleep_duration_diff_mean3d', 'sleep_time_ratio_mean3d', 'wake_time_ratio_mean3d', 'sleep_duration_ratio_mean3d', 'sleep_time_std3d', 'wake_time_std3d', 'sleep_duration_min_std3d', 'sleep_time_diff_std3d', 'wake_time_diff_std3d', 'sleep_duration_diff_std3d', 'sleep_time_ratio_std3d', 'wake_time_ratio_std3d', 'sleep_duration_ratio_std3d', 'sleep_time_mean5d', 'wake_time_mean5d', 'sleep_duration_min_mean5d', 'sleep_time_diff_mean5d', 'wake_time_diff_mean5d', 'sleep_duration_diff_mean5d', 'sleep_time_ratio_mean5d', 'wake_time_ratio_mean5d', 'sleep_duration_ratio_mean5d', 'sleep_time_std5d', 'wake_time_std5d', 'sleep_duration_min_std5d', 'sleep_time_diff_std5d', 'wake_time_diff_std5d', 'sleep_duration_diff_std5d', 'sleep_time_ratio_std5d', 'wake_time_ratio_std5d', 'sleep_duration_ratio_std5d', 'sleep_time_mean7d', 'wake_time_mean7d', 'sleep_duration_min_mean7d', 'sleep_time_diff_mean7d', 'wake_time_diff_mean7d', 'sleep_duration_diff_mean7d', 'sleep_time_ratio_mean7d', 'wake_time_ratio_mean7d', 'sleep_duration_ratio_mean7d', 'sleep_time_std7d', 'wake_time_std7d', 'sleep_duration_min_std7d', 'sleep_time_diff_std7d', 'wake_time_diff_std7d', 'sleep_duration_diff_std7d', 'sleep_time_ratio_std7d', 'wake_time_ratio_std7d', 'sleep_duration_ratio_std7d', 'weekday_avg_sleep', 'sleep_duration_weekday_avg_diff', 'sleep_duration_weekday_avg_div']
# drop_features = [i for i in feats if i in train.columns]
# train = train.drop(columns=drop_features)
# train = train.merge(mScreenStatus_llm,on=['subject_id','lifelog_date'],how='left')
# test = test.drop(columns=drop_features)
# test = test.merge(mScreenStatus_llm,on=['subject_id','lifelog_date'],how='left')

# check 
print('# train  shape:',train.shape)
print('# test   shape:',test.shape)

# train  shape: (450, 259)
# test   shape: (250, 259)


In [5]:
# 숫자형 컬럼만 선택해서 결측값 -1로 채우기
train[train.select_dtypes(include='number').columns] = train.select_dtypes(include='number').fillna(-1)
test[test.select_dtypes(include='number').columns] = test.select_dtypes(include='number').fillna(-1)

# LLM 설정

In [6]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
llm_model_path = "Qwen/Qwen3-0.6B"

In [8]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],#Best
    #target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"],
)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [10]:
tokenizer = AutoTokenizer.from_pretrained(llm_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [11]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df.index)

    def __getitem__(self, idx):
        return np.array([idx])

In [12]:
def train_iter(model, loader, optimizer, criterion, df_clean):
    model.train()
    
    total_loss = 0
    correct = 0
    pred = []
    label = []
    for batchIdx, sampledIdx in enumerate(tqdm(loader, position=0, leave=True)):
        #print(data)
        
        optimizer.zero_grad()
        
        #text
        sampledIdx = sampledIdx.cpu().data.numpy()
        sampledRowText = list(df_clean["text"].iloc[list(sampledIdx.flatten())])
        #label
        sampledRowLabels = torch.tensor(list(df_clean["label"].iloc[list(sampledIdx.flatten())])).to(device)
        
        #encoded
        encoded_input = tokenizer(sampledRowText, truncation=True, padding=True, return_tensors='pt').to(device) # Output shape: [bs, num_Labels]
        encoded_inputIds = encoded_input["input_ids"].to(device)
        encoded_attnMask = encoded_input["attention_mask"].to(device)

        #model
        outputs = model(input_ids=encoded_inputIds, attention_mask=encoded_attnMask)
        #print(outputs)

        # label type change
        sampledRowLabels = sampledRowLabels.to(outputs.logits.device).long()  # shape: [1]
        #print(outputs.logits)
        # loss
        loss = criterion(outputs.logits, sampledRowLabels) 
        total_loss += loss.item()
        
        #acurracy
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(outputs.logits, dim=-1)
        pred.extend(predicted_class.flatten().cpu().data.numpy())
        label.extend(sampledRowLabels.cpu().data.numpy())     

        #back propagation
        loss.backward()
        optimizer.step()

    train_accuracy_score = metrics.f1_score(label,pred, average='macro')
    return total_loss / len(loader), train_accuracy_score

In [13]:
def valid_iter(model, valid_loader, criterion, df_clean):
    model.eval()
    with torch.no_grad():
        
        total_loss = 0
        correct = 0

        pred = []
        label = []

        for batchIdx, sampledIdx in enumerate(tqdm(valid_loader, position=0, leave=True)):
            
            sampledRowText = list(df_clean["text"].iloc[list(sampledIdx.flatten())])
            sampledRowLabels = torch.tensor(list(df_clean["label"].iloc[list(sampledIdx.flatten())]))

            encoded_input = tokenizer(sampledRowText, truncation=True, padding=True, return_tensors='pt').to(device) # Output shape: [bs, num_Labels]
            encoded_inputIds = encoded_input["input_ids"].to(device)
            encoded_attnMask = encoded_input["attention_mask"].to(device)
            
            #model
            outputs = model(input_ids=encoded_inputIds, attention_mask=encoded_attnMask)
    
            # label type change
            sampledRowLabels = sampledRowLabels.to(outputs.logits.device).long()  # shape: [1]
            
            # loss
            loss = criterion(outputs.logits, sampledRowLabels) 
            total_loss += loss.item()
            
            #acurracy
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(outputs.logits, dim=-1)
            pred.extend(predicted_class.flatten().cpu().data.numpy())
            label.extend(sampledRowLabels.cpu().data.numpy())     

        valid_accuracy_score = metrics.f1_score(label,pred, average='macro')
    return total_loss/len(valid_loader), valid_accuracy_score

In [14]:
model_loading_path = "/kaggle/working/"

def inference(model, loader, test_data, col):
    #best model 불러와서 call 하기
    model.load_state_dict(torch.load(model_loading_path + col+"_"+best_model_path))
    model.eval()
    
    preds = []
    preds_prob = []
    with torch.no_grad():
        total_loss = 0
        correct = 0

        pred = []
        label = []
        
        for batchIdx, sampledIdx in enumerate(tqdm(loader, position=0, leave=True)):
            
            sampledRowText = list(test_data["text"].iloc[list(sampledIdx.flatten())])

            encoded_input = tokenizer(sampledRowText, truncation=True, padding=True, return_tensors='pt').to("cuda") # Output shape: [bs, num_Labels]
            encoded_inputIds = encoded_input["input_ids"].to("cuda")
            encoded_attnMask = encoded_input["attention_mask"].to("cuda")
            
            outputs = model(input_ids=encoded_inputIds, attention_mask=encoded_attnMask)
            logits = outputs.logits


            #acurracy
            probs = torch.nn.functional.softmax(outputs.logits.cpu(), dim=-1)
            
            #확률구하기
            preds_prob.extend(probs.cpu().data.numpy())

            #acurracy
            pred.extend(torch.argmax(logits, dim=1).flatten().cpu().data.numpy())
            
    return pred, preds_prob

In [15]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha  # optional: list or tensor of class weights
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)  # prevents nans when probability is 0
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [16]:
# Define your training loop
epochs = 16
total_loss = 0
correct = 0

#best model 를 찾기 위한
best = np.inf

#3번까지 validation loss 터지면 stop 시킴
patience = 3

#best model save
#result_path = "/kaggle/working/"
#best_model_path = os.path.join(result_path, 'best_model.pt')
best_model_path = 'best_model.pt'
bad_counter = 0

In [17]:
# 피처 추가
X_Feature = {
    "Q1": ["Q1_te2", "wake_time_ratio", "mlight_first_wakeup_minutes", "wake_time_diff", "Q1_te", "lights_off_time", "sleep_duration_ratio", "active_hour_mean_speed", "beforebed_통화_time", "rolling_sleep_time_2d", "activehour_NAVER_time"],
    "Q2": ["Q2_te", "Q2_te2", "activehour_total_screen_time", "beforebed_unique_bssid_count", "wake_time_lag1", "light_rolling_wake_time_2d", "beforebed_max_rssi", "active_hour_std_hr", "beforebed_top_bssid_count", "activehour_screen_time_vs_avg_pct", "activehour_메신저_time"],
    "Q3": ["Q3_te2", "light_sleep_time_lag2", "mlight_first_wakeup_minutes", "rolling_sleep_time_3d", "light_rolling_sleep_duration_3d", "Q3_te", "beforebed_scan_count", "active_hour_distance_x", "activehour_통화_time", "walking_minutes"],
    "S1": ["S1_te", "wake_time_diff", "S1_te2", "sleep_duration_ratio", "m_activity_met@240min@sum@04h00m", "beforebed_screen_time_vs_avg_pct", "wake_time_ratio", "rolling_wake_time_3d", "m_activity_0@240min@std@20h00m", "m_activity@240min@std@12h00m", "beforebed_메신저_time", "sleep_duration_diff", "light_sleep_time_diff", "active_hour_min_hr"],
    "S2": ["S2_te2", "S2_te", "light_sleep_time_lag1", "work_hour_unknown_ratio", "m_activity@240min@std@12h00m", "beforebed_strong_signal_ratio", "light_rolling_wake_time_2d", "free_hour_rssi_mean", "activehour_전화_time", "sleep_hour_mean_speed","activehour_screen_time_vs_avg_pct","light_wake_time_diff_lag2","beforebed_max_rssi","avg_charging_duration","m_activity_met@240min@std@12h00m"],
    "S3": ["S3_te", "S3_te2", "beforebed_메신저_time", "light_wake_time_diff", "sleep_time_diff_lag1", "light_sleep_time_lag2", "m_activity_met@240min@sum@16h00m", "free_hour_rssi_max", "light_weekday_avg_sleep", "불끈시간부터기상시간","sleep_hour_distance_x","activehour_scan_count"]
}

In [18]:
X_Feature_alias = {
    "Q1": [
        "Q1_encoded_time_2",
        "wake_time_to_baseline_ratio",
        "minutes_to_first_wake_after_light",
        "wake_time_difference",
        "Q1_encoded_time",
        "lights_off_clock_time",
        "sleep_duration_ratio_to_guideline",
        "mean_speed_during_active_hours",
        "call_duration_before_bed",
        "sleep_time_rolling_avg_2d",
        "NAVER_time_active_hours"
    ],
    "Q2": [
        "Q2_encoded_time",
        "Q2_encoded_time_2",
        "total_screen_time_active_hours",
        "unique_wifi_count_before_bed",
        "previous_day_wake_time",
        "light_based_wake_time_rolling_avg_2d",
        "max_wifi_signal_before_bed",
        "std_heart_rate_active_hours",
        "frequent_wifi_count_before_bed",
        "screen_time_vs_avg_pct_active_hours",
        "messenger_usage_time_active_hours"
    ],
    "Q3": [
        "Q3_encoded_time_2",
        "light_sleep_duration_lag2",
        "minutes_to_first_wake_after_light",
        "sleep_time_rolling_avg_3d",
        "light_based_sleep_duration_rolling_avg_3d",
        "Q3_encoded_time",
        "wifi_scan_count_before_bed",
        "distance_traveled_active_hours",
        "call_duration_active_hours",
        "total_walking_minutes"
    ],
    "S1": [
        "S1_encoded_time",
        "wake_time_difference",
        "S1_encoded_time_2",
        "sleep_duration_ratio_to_guideline",
        "met_sum_0to4am",
        "screen_time_vs_avg_pct_before_bed",
        "wake_time_to_baseline_ratio",
        "wake_time_rolling_avg_3d",
        "activity_std_8pm_to_midnight",
        "activity_std_12pm_to_4pm"
        "messenger_usage_time_before_bed",
        "sleep_time_difference",
        "light_based_sleep_time_difference",
        "activity_time_mininum_hours"
        
    ],
    "S2": [
        "S2_encoded_time_2",
        "S2_encoded_time",
        "light_sleep_duration_lag1",
        "unknown_activity_ratio_work_hours",
        "activity_std_12pm_to_4pm",
        "strong_wifi_signal_ratio_before_bed",
        "light_based_wake_time_rolling_avg_2d",
        "avg_wifi_signal_strength_free_hours",
        "phone_call_time_active_hours",
        "mean_movement_speed_sleep_hours",
        "activity_time_screen_hour_vs_avg_pct","light_wake_time_diff_lag2","beforebed_max_rssi","avg_charging_duration","m_activity_met@240min@std@12h00m"
    ],
    "S3": [
        "S3_encoded_time",
        "S3_encoded_time_2",
        "messenger_usage_time_before_bed",
        "light_based_wake_time_difference",
        "sleep_time_difference_lag1",
        "light_sleep_duration_lag2",
        "met_sum_4pm_to_8pm",
        "max_wifi_signal_strength_free_hours",
        "light_based_weekday_avg_sleep_duration",
        "time_from_first_movement_to_final_wake",
        "sleep_hour_distance_x","active_hour_scan_count"
    ]
}

In [19]:
Prompt = {
    "Q1": "Based on the following sleep-related information, classify the overall perceived sleep quality upon waking as either below or above the individual’s average. (0: Below average, 1: Above average) ",
    "Q2": "Using the following data, determine the level of physical fatigue the individual felt before sleep. (0: High fatigue, 1: Low fatigue) ",
    "Q3": "Based on the information below, classify the stress level the individual experienced before going to bed. (0: High stress, 1: Low stress) ",
    "S1": "Based on the provided information, classify how well the individual's total sleep time (TST) aligns with recommended sleep guidelines. (0: Not met, 1: Partially met, 2: Fully met) ",
    "S2": "Using the following indicators, determine whether the individual met the guideline for sleep efficiency (SE). (0: Not met, 1: Met) ",
    "S3": "From the data provided, assess if the sleep onset latency (SOL) guideline was met. (0: Not met, 1: Met) "
}

# 재현성 확보를 위한 시드 고정

In [20]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"

In [21]:
# https://github.com/huggingface/transformers/blob/f4fc42216cd56ab6b68270bf80d811614d8d59e4/src/transformers/trainer_utils.py#L93
def enable_full_determinism(seed: int, warn_only: bool = True):
    """
    Helper function for reproducible behavior during distributed training. See
    - https://pytorch.org/docs/stable/notes/randomness.html for pytorch
    - https://www.tensorflow.org/api_docs/python/tf/config/experimental/enable_op_determinism for tensorflow
    """

    os.environ["PYTHONHASHSEED"] = str(seed)
    # set seed first
    set_seed(seed)

    # Enable PyTorch deterministic mode. This potentially requires either the environment
    # variable 'CUDA_LAUNCH_BLOCKING' or 'CUBLAS_WORKSPACE_CONFIG' to be set,
    # depending on the CUDA version, so we set them both here
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    # The environment variable required to enable deterministic mode on Ascend NPUs.
    os.environ["ASCEND_LAUNCH_BLOCKING"] = "1"
    os.environ["HCCL_DETERMINISTIC"] = "1"

    os.environ["FLASH_ATTENTION_DETERMINISTIC"] = "1"
    torch.use_deterministic_algorithms(True, warn_only=warn_only)

    # Enable CUDNN deterministic mode
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_seed(seed: int, deterministic: bool = True):
    """
    Helper function for reproducible behavior to set the seed in `random`, `numpy`, `torch` and/or `tf` (if installed).

    Args:
        seed (`int`):
            The seed to set.
        deterministic (`bool`, *optional*, defaults to `False`):
            Whether to use deterministic algorithms where available. Can slow down training.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True)

In [22]:
SEED = 42
enable_full_determinism(SEED)  # 42는 원하는 시드값

In [23]:
generator = torch.Generator()
generator.manual_seed(SEED)  # 시드 고정

### ============================

### run_basemodel

In [24]:
#epochs = 1
def run_basemodel(train, test, valid_ids, common_params, n_splits, random_state=42):

    lgb_A = 0.3
    xgb_B = 0.3
    tab_C = 0.1
    llm_D = 0.3

    train_df = train.copy()
    test_df = test.copy()

    submission_final = test_df[['subject_id', 'sleep_date', 'lifelog_date']].copy()
    submission_final['lifelog_date'] = pd.to_datetime(submission_final['lifelog_date']).dt.date

    # 타겟
    targets_binary = ['Q1', 'Q2', 'Q3', 'S2', 'S3']
    targets_binary_name = ['기상직후수면질','취침전신체적피로','취침전스트레스','수면효율','수면잠들기시간']
    target_multiclass = 'S1'
    all_targets = targets_binary + [target_multiclass]

    # 노이즈 수준 설정
    def add_noise(series, noise_level, seed=3):
        rng = np.random.default_rng(seed)
        return series * (1 + noise_level * rng.standard_normal(len(series)))

    noise_level = 0.015  # 필요에 따라 조정

    # 타겟인코딩
    for tgt in all_targets:

      encoder_feats = ['subject_id','month','weekend'] # 'weekday', 'subject_id','month','weekend'

      #### 타겟인코딩1

      subject_mean = train_df.groupby(encoder_feats)[tgt].mean().rename(f'{tgt}_te')
      train_df = train_df.merge(subject_mean, on=encoder_feats, how='left')
      test_df = test_df.merge(subject_mean, on=encoder_feats, how='left')
      global_mean = train_df[tgt].mean()
      test_df[f'{tgt}_te'] = test_df[f'{tgt}_te'].fillna(global_mean)

      # 노이즈 추가
      train_df[f'{tgt}_te'] = add_noise(train_df[f'{tgt}_te'], noise_level)
      test_df[f'{tgt}_te'] = add_noise(test_df[f'{tgt}_te'], noise_level)

      #### 타겟인코딩2

      # 새로운 범주형 열 생성
      train_df['TMP'] = train_df[encoder_feats].applymap(str).apply(lambda x: ''.join(x) ,axis=1)
      test_df['TMP'] = test_df[encoder_feats].applymap(str).apply(lambda x: ''.join(x) ,axis=1)

      # 인코더
      encoder = TargetEncoder(cols=['TMP'], smoothing=300) # 40
      encoder.fit(train_df[['TMP']], train_df[tgt])

      # 인코딩 결과를 새로운 열에 저장
      train_df[f'{tgt}_te2'] = encoder.transform(train_df[['TMP']])
      test_df[f'{tgt}_te2'] = encoder.transform(test_df[['TMP']])

      # 노이즈 추가
      train_df[f'{tgt}_te2'] = add_noise(train_df[f'{tgt}_te2'], noise_level)
      test_df[f'{tgt}_te2'] = add_noise(test_df[f'{tgt}_te2'], noise_level)

      # 불필요한 변수 제거
      train_df = train_df.drop(columns=['TMP'])
      test_df = test_df.drop(columns=['TMP'])


    # 인코딩
    PK = ['sleep_date', 'lifelog_date', 'subject_id']
    encoder = LabelEncoder()
    categorical_features = [i for i in train_df.select_dtypes(include=['object', 'category']).columns if i not in PK+['pk']]
    for col in categorical_features:
        print(col)
        train_df[col] = encoder.fit_transform(train_df[col])
        test_df[col] = encoder.fit_transform(test_df[col])

    # X
    X = train_df.drop(columns=PK + all_targets)
    test_X = test_df.drop(columns=PK + all_targets)
    print(f'# X shape: {X.shape}')
    print(f'# test_X shape: {test_X.shape}')

    print('\n STEP1: 실험 결과 확인')
    print("=============== Validation Results ==============")
    total_avg_f1s = []
    val_f1 = []
    binary_val_preds = {}
    multiclass_val_preds = {}
    binary_test_preds = {}
    multiclass_test_preds = {}
    test_preds = {}

    # Find optimal weights
    best_weights = []
    best_scores = []


    for col in targets_binary:
        # binary
        y = train_df[col]

        valid_ids['pk'] = valid_ids['subject_id']+valid_ids['sleep_date']
        train_df['pk'] = train_df['subject_id']+train_df['sleep_date']

        X_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
        X_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
        y_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()
        y_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()

        def prepare_input_text(row):
            target_cols = ['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']
            feature_cols = [col for col in X_Feature[col] if col not in target_cols]

            # 프롬프트 시작 문장에 결측치 설명 추가
            base_prompt = Prompt["S1"] if is_multiclass else Prompt[col]
            instruct_txt = base_prompt.strip() + "\n(Note: -1.0 indicates missing value in the data)\n###DATA###\n"
  
            if is_multiclass:
                for orig_name, alias_name in zip(X_Feature["S1"], X_Feature_alias["S1"]):
                    if orig_name not in target_cols:
                        instruct_txt += f"{alias_name} : {row[orig_name]}, "
            else:
                for orig_name, alias_name in zip(X_Feature[col], X_Feature_alias[col]):
                    if orig_name not in target_cols:
                        instruct_txt += f"{alias_name} : {row[orig_name]}, "
                
            return instruct_txt.strip()[:-1]+'.' 


        ######## llm train start #########
        is_multiclass = False
        llm_model = AutoModelForSequenceClassification.from_pretrained(llm_model_path, num_labels=2, torch_dtype=torch.float16, device_map='auto')
        lora_model = get_peft_model(llm_model, config)
        
        optimizer = torch.optim.AdamW(llm_model.parameters(), lr=3e-4, weight_decay=1e-4)
        #optimizer = torch.optim.AdamW(llm_model.parameters(), lr=1e-5, weight_decay=1e-4)
        # alpha = torch.tensor([1.048, 0.670, 1.807], dtype=torch.float16, device=device)
        # criterion = FocalLoss(gamma=2.0,alpha=alpha)
        criterion = FocalLoss(gamma=2.0)
    
        # Add learning rate scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=2,
            verbose=True,
            min_lr=1e-6
        )
        
        total_loss = 0
        correct = 0
        bad_counter = 0
        
        #best model 를 찾기 위한
        best = np.inf
    
        X_train_llm = X_train.copy()
        X_valid_llm = X_valid.copy()
    
        # X_train_llm = X_train_llm[:15][X_Feature[col]]
        # print(X_train_llm.head(1))
        # X_valid_llm = X_valid_llm[:15][X_Feature[col]]
        # print(X_valid_llm.head(1))
    
        X_train_llm = X_train_llm[X_Feature[col]]
        X_valid_llm = X_valid_llm[X_Feature[col]]
    
        X_train_llm['input'] = X_train_llm.apply(lambda x: prepare_input_text(x),axis=1)
        #print(X_train_llm.head(1))
        X_valid_llm['input'] = X_valid_llm.apply(lambda x: prepare_input_text(x),axis=1)
        
        # X_train_llm["label"] = y_train[:15]
        # X_valid_llm["label"] = y_valid[:15]
        X_train_llm["label"] = y_train
        X_valid_llm["label"] = y_valid
        X_train_llm["text"] = X_train_llm["input"]
        X_valid_llm["text"] = X_valid_llm["input"]
                
        training_data = CustomDataset(X_train_llm)
        validation_data = CustomDataset(X_valid_llm)
        
        #train_dataloader = DataLoader(training_data, batch_size=1, shuffle=True,worker_init_fn=seed_worker, generator=g)
        #train_dataloader = DataLoader(training_data, batch_size=1, shuffle=True)
        train_dataloader = DataLoader(training_data, batch_size=1, shuffle=True,generator=generator)
        val_dataloader = DataLoader(validation_data, batch_size=1, shuffle=False)
        
    
        X_train_llm_clean = X_train_llm
        X_valid_llm_clean = X_valid_llm
    
        
        for epoch in range(epochs):
            avg_train_loss, avg_train_acc  = train_iter(llm_model, train_dataloader, optimizer, criterion, X_train_llm_clean)
            
            avg_vaild_loss, avg_vaild_acc = valid_iter(llm_model, val_dataloader, criterion, X_valid_llm_clean)
    
            # Update learning rate based on validation loss
            scheduler.step(avg_vaild_loss)
            
            if avg_vaild_loss < best:
                best = avg_vaild_loss
                torch.save(llm_model.cpu().state_dict(), col+"_" + best_model_path)
                llm_model.cuda()
                bad_counter = 0
            else:
                bad_counter += 1
        
            if bad_counter == patience:
                break
            
            print(f'{col} Epoch: {str(epoch+1)}: t_loss:{avg_train_loss:.3f} t_acc:{avg_train_acc:.3f} v_loss:{avg_vaild_loss:.3f} v_acc:{avg_vaild_acc:.3f}')
            
            wandb.log({"train": {'epoch': epoch, "acc": avg_train_acc, "loss": avg_train_loss}, "val": {'epoch': epoch, "acc": avg_vaild_acc, "loss": avg_vaild_loss}})
            
            # ✅ 메모리 정리
            del avg_train_loss, avg_train_acc, avg_vaild_loss, avg_vaild_acc
            gc.collect()
            torch.cuda.empty_cache()         
        ######## llm train end #################

        # Get parameters for both models
        lgb_params = common_params[col].copy()
        lgb_params['random_state'] = random_state

        xgb_params = {
            'n_estimators': 1000,
            'learning_rate': 0.01,
            'max_depth': 6,
            'min_child_weight': 1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state
        }

        # Train LightGBM
        lgb_model = LGBMClassifier(**lgb_params)
        lgb_model.fit(X_train, y_train)

        # Train XGBoost
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(X_train, y_train)

        tabpfn_params = {
            'device': 'cuda'
        }

        # Train TabPFN
        tabpfn_model = TabPFNClassifier(**tabpfn_params)
        tabpfn_model.fit(X_train, y_train)

        # Prediction
        tab_pred_valid = tabpfn_model.predict_proba(X_valid.values)[:, 1]
        lgb_pred_valid = lgb_model.predict_proba(X_valid)[:, 1]
        xgb_pred_valid = xgb_model.predict_proba(X_valid)[:, 1]
        _, llm_pred_valid = inference(llm_model, val_dataloader, X_valid_llm, col)
        llm_pred_valid = np.array([arr[1] for arr in llm_pred_valid], dtype=np.float32)

        pred_valid = (lgb_A * lgb_pred_valid + xgb_B * xgb_pred_valid + tab_C * tab_pred_valid + llm_D * llm_pred_valid  > 0.5).astype(int)

        f1 = f1_score(y_valid, pred_valid, average='macro')
        val_f1.append(f1)

        # Store predictions
        binary_val_preds[col] = {
            'lgb': lgb_pred_valid,
            'xgb': xgb_pred_valid,
            'tab': tab_pred_valid,
            'llm': llm_pred_valid,
            'true': y_valid
        }

    # multiclass
    y = train_df[target_multiclass]
    X_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
    X_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
    y_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()
    y_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()


    ######## llm train start #################
    is_multiclass = True
    llm_model = AutoModelForSequenceClassification.from_pretrained(llm_model_path, num_labels=3, torch_dtype=torch.float16, device_map='auto')
    lora_model = get_peft_model(llm_model, config)
    
    optimizer = torch.optim.AdamW(llm_model.parameters(), lr=3e-4, weight_decay=1e-4)
    #optimizer = torch.optim.AdamW(llm_model.parameters(), lr=1e-5, weight_decay=1e-4)
    # alpha = torch.tensor([1.048, 0.670, 1.807], dtype=torch.float16, device=device)
    # criterion = FocalLoss(gamma=2.0,alpha=alpha)
    criterion = FocalLoss(gamma=2.0)

    # Add learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2,
        verbose=True,
        min_lr=1e-6
    )
    
    total_loss = 0
    correct = 0
    bad_counter = 0
    
    #best model 를 찾기 위한
    best = np.inf

    X_train_llm = X_train.copy()
    X_valid_llm = X_valid.copy()

    # X_train_llm = X_train_llm[:15][X_Feature[col]]
    # print(X_train_llm.head(1))
    # X_valid_llm = X_valid_llm[:15][X_Feature[col]]
    # print(X_valid_llm.head(1))

    X_train_llm = X_train_llm[X_Feature["S1"]]
    X_valid_llm = X_valid_llm[X_Feature["S1"]]

    X_train_llm['input'] = X_train_llm.apply(lambda x: prepare_input_text(x),axis=1)
    #print(X_train_llm.head(1))
    X_valid_llm['input'] = X_valid_llm.apply(lambda x: prepare_input_text(x),axis=1)
    
    # X_train_llm["label"] = y_train[:15]
    # X_valid_llm["label"] = y_valid[:15]
    X_train_llm["label"] = y_train
    X_valid_llm["label"] = y_valid
    X_train_llm["text"] = X_train_llm["input"]
    X_valid_llm["text"] = X_valid_llm["input"]
            
    training_data = CustomDataset(X_train_llm)
    validation_data = CustomDataset(X_valid_llm)
    
    train_dataloader = DataLoader(training_data, batch_size=1, shuffle=True,generator=generator)
    val_dataloader = DataLoader(validation_data, batch_size=1, shuffle=False)

    X_train_llm_clean = X_train_llm
    X_valid_llm_clean = X_valid_llm

    
    for epoch in range(epochs):
        avg_train_loss, avg_train_acc  = train_iter(llm_model, train_dataloader, optimizer, criterion, X_train_llm_clean)
        
        avg_vaild_loss, avg_vaild_acc = valid_iter(llm_model, val_dataloader, criterion, X_valid_llm_clean)

        # Update learning rate based on validation loss
        scheduler.step(avg_vaild_loss)
        
        if avg_vaild_loss < best:
            best = avg_vaild_loss
            torch.save(llm_model.cpu().state_dict(), "S1_" + best_model_path)
            llm_model.cuda()
            bad_counter = 0
        else:
            bad_counter += 1
    
        if bad_counter == patience:
            break
        
        print(f'S1 Epoch: {str(epoch+1)}: t_loss:{avg_train_loss:.3f} t_acc:{avg_train_acc:.3f} v_loss:{avg_vaild_loss:.3f} v_acc:{avg_vaild_acc:.3f}')
        
        wandb.log({"train": {'epoch': epoch, "acc": avg_train_acc, "loss": avg_train_loss}, "val": {'epoch': epoch, "acc": avg_vaild_acc, "loss": avg_vaild_loss}})
        
        # ✅ 메모리 정리
        del avg_train_loss, avg_train_acc, avg_vaild_loss, avg_vaild_acc
        gc.collect()
        torch.cuda.empty_cache()         
    ######## llm train end #################

    # Get parameters for both models
    lgb_params = common_params['S1'].copy()
    lgb_params['random_state'] = random_state

    xgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': random_state
    }

    # 클래스 weight 계산
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))

    # 각 샘플에 대해 weight 매핑
    w_train = pd.Series(y_train).map(class_weights)
    w_train = compute_sample_weight(class_weight='balanced', y=y_train)

    # Train LightGBM
    lgb_model = LGBMClassifier(**lgb_params, objective='multiclass', num_class=3)
    lgb_model.fit(X_train, y_train, sample_weight=w_train)

    # Train XGBoost
    xgb_model = XGBClassifier(**xgb_params, objective='multi:softmax', num_class=3)
    xgb_model.fit(X_train, y_train,sample_weight=w_train)

    tabpfn_params = {
        'device': 'cuda'
    }

    # Train TabPFN
    tabpfn_model = TabPFNClassifier(**tabpfn_params)
    tabpfn_model.fit(X_train, y_train)

    # Get predictions and ensemble
    _, llm_pred_valid = inference(llm_model, val_dataloader, X_valid_llm, "S1")
    lgb_pred_valid = lgb_model.predict_proba(X_valid)
    xgb_pred_valid = xgb_model.predict_proba(X_valid)
    tab_pred_valid = tabpfn_model.predict_proba(X_valid.values)
    llm_pred_valid = np.array(llm_pred_valid, dtype=np.float32)


    pred_valid = np.argmax(lgb_A * lgb_pred_valid + xgb_B * xgb_pred_valid + tab_C * tab_pred_valid + llm_D * llm_pred_valid, axis=1)

    f1 = f1_score(y_valid, pred_valid, average='macro')
    val_f1.append(f1)

    multiclass_val_preds = {
        'lgb': lgb_pred_valid,
        'xgb': xgb_pred_valid,
        'tab': tab_pred_valid,
        'llm': llm_pred_valid,
        'true': y_valid
    }

    # Generate all possible weight combinations that sum to 1
    step = 0.1
    for lgb_A in np.arange(0, 1.1, step):
        for xgb_B in np.arange(0, 1.1 - lgb_A, step):
            for tab_C in np.arange(0, 1.1 - lgb_A - xgb_B, step):
                llm_D = 1 - (lgb_A + xgb_B + tab_C)
                if llm_D >= 0:
                    weights = (lgb_A, xgb_B, tab_C, llm_D)
                    print("========================================")
                    print(f"\nTrying weights: lgb_A={lgb_A:.1f}, xgb_B={xgb_B:.1f}, tab_C={tab_C:.1f}, llm_D={llm_D:.1f}")
                    
                    # Calculate validation score with current weights
                    val_scores = []
                    
                    # Binary targets
                    for col in targets_binary:
                        preds = binary_val_preds[col]
                        
                        ensemble_pred = (lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + 
                                      tab_C * preds['tab'] + llm_D * preds['llm'] > 0.5).astype(int)
                        f1 = f1_score(preds['true'], ensemble_pred, average='macro')
                        val_scores.append(f1)
                        print(f" Validation Score {col}:{f1:.4f}")
                        
                    # Multiclass target
                    preds = multiclass_val_preds
                    ensemble_pred = np.argmax(lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + 
                                           tab_C * preds['tab'] + llm_D * preds['llm'], axis=1)
                    f1 = f1_score(preds['true'], ensemble_pred, average='macro')
                    print(f" Validation Score S1:{f1:.4f}")
                    val_scores.append(f1)
                    
                    avg_score = np.mean(val_scores)
                    best_weights.append(weights)
                    best_scores.append(avg_score)
                    
                    print(f"Average Validation Score: {avg_score:.4f}")

    # Sort results and get all
    sorted_indices = np.argsort(best_scores)[::-1]
    all_weights = [best_weights[i] for i in sorted_indices]
    all_scores = [best_scores[i] for i in sorted_indices]

    print("\nTop All Weight Combinations:")
    for i, (weights, score) in enumerate(zip(all_weights, all_scores)):
        print(f"Rank {i+1}: lgb_A={weights[0]:.1f}, xgb_B={weights[1]:.1f}, tab_C={weights[2]:.1f}, llm_D={weights[3]:.1f} - Score: {score:.4f}")

    avg_f1 = np.mean(val_f1)
    total_avg_f1s.append(avg_f1)
    detail = " ".join([f"{name}({tname}):{score:.4f}" for name, tname, score in zip(targets_binary + [target_multiclass], targets_binary_name + ['S1'], val_f1)])
          
    print(f" 평균 F1: {avg_f1:.4f} / [상세] {detail}")
    print(f"# 전체 평균 F1: {np.mean(total_avg_f1s):.4f}")
    print("================================================")

    # ------------------------------------------ modeling with 100% train & no valid --------------------------------------------------------------------
    
    print('\n STEP2: 전체 데이터로 모델 재학습')
    print("====== modeling with 100% train & no valid =====")

    # binary
    binary_preds = {}
    binary_preds_proba = {}
    for col in targets_binary:
        # Get parameters for both models
        lgb_params = common_params[col].copy()
        lgb_params['random_state'] = random_state

        xgb_params = {
            'n_estimators': 1000,
            'learning_rate': 0.01,
            'max_depth': 6,
            'min_child_weight': 1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state
        }

        y = train_df[col]

        is_multiclass = False

        # LLM Valid Model load and Inference
        test_X_llm = test_X.copy()
        test_X_llm = test_X_llm[X_Feature[col]]
        test_X_llm['input'] = test_X_llm.apply(lambda x: prepare_input_text(x),axis=1)
        test_X_llm["text"] = test_X_llm["input"]
        test_data = CustomDataset(test_X_llm)
        test_dataloader = DataLoader(test_data, batch_size=1, shuffle=False)
        
        llm_model = AutoModelForSequenceClassification.from_pretrained(llm_model_path, num_labels=2, torch_dtype=torch.float16, device_map='auto')
        lora_model = get_peft_model(llm_model, config)
        _, llm_pred = inference(llm_model, test_dataloader, test_X_llm, col)
        llm_pred = np.array([arr[1] for arr in llm_pred], dtype=np.float32)    
    
        # Train LightGBM
        lgb_model = LGBMClassifier(**lgb_params)
        lgb_model.fit(X, y)

        # Train XGBoost
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(X, y)

        tabpfn_params = {
            'device': 'cuda'
        }

        # Train TabPFN
        tabpfn_model = TabPFNClassifier(**tabpfn_params)
        tabpfn_model.fit(X, y)

        tab_pred = tabpfn_model.predict_proba(test_X)[:, 1]
        lgb_pred = lgb_model.predict_proba(test_X)[:, 1]
        xgb_pred = xgb_model.predict_proba(test_X)[:, 1]

        binary_preds[col] = (lgb_A * lgb_pred + xgb_B * xgb_pred + tab_C * tab_pred + llm_D * llm_pred > 0.5).astype(int)

        # Store predictions
        binary_test_preds[col] = {
            'lgb': lgb_pred,
            'xgb': xgb_pred,
            'tab': tab_pred,
            'llm': llm_pred,
        }

        # Feature importance (using LightGBM's importance)
        fi_df = pd.DataFrame({'feature': X.columns, 'importance': lgb_model.feature_importances_})
        top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
        feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
        print(f"[{col}] {feat_str}")

    # multiclass
    y = train_df['S1']

    # Get parameters for both models
    lgb_params = common_params['S1'].copy()
    lgb_params['random_state'] = random_state

    xgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': random_state
    }

    # 클래스 weight 계산
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    class_weights = dict(zip(classes, weights))

    # 각 샘플에 대해 weight 매핑
    w_train = pd.Series(y).map(class_weights)
    w_train = compute_sample_weight(class_weight='balanced', y=y)

    is_multiclass = True
    
     # LLM Valid Model load and Inference
    test_X_llm = test_X.copy()
    test_X_llm = test_X_llm[X_Feature["S1"]]
    test_X_llm['input'] = test_X_llm.apply(lambda x: prepare_input_text(x),axis=1)
    test_X_llm["text"] = test_X_llm["input"]
    test_data = CustomDataset(test_X_llm)
    test_dataloader = DataLoader(test_data, batch_size=1, shuffle=False)

    llm_model = AutoModelForSequenceClassification.from_pretrained(llm_model_path, num_labels=3, torch_dtype=torch.float16, device_map='auto')
    lora_model = get_peft_model(llm_model, config)
    _, llm_pred = inference(llm_model, test_dataloader, test_X_llm, "S1")
    llm_pred = np.array(llm_pred, dtype=np.float32)
    
    # Train LightGBM
    lgb_model = LGBMClassifier(**lgb_params, objective='multiclass', num_class=3)
    lgb_model.fit(X, y, sample_weight=w_train)

    # Train XGBoost
    xgb_model = XGBClassifier(**xgb_params, objective='multi:softmax', num_class=3)
    xgb_model.fit(X, y, sample_weight=w_train)

    tabpfn_params = {
        'device': 'cuda'
    }

     # Train TabPFN
    tabpfn_model = TabPFNClassifier(**tabpfn_params)
    tabpfn_model.fit(X, y)

    # Get predictions and ensemble
    lgb_pred = lgb_model.predict_proba(test_X)
    xgb_pred = xgb_model.predict_proba(test_X)
    tab_pred = tabpfn_model.predict_proba(test_X)

    multiclass_test_preds = {
        'lgb': lgb_pred,
        'xgb': xgb_pred,
        'tab': tab_pred,
        'llm': llm_pred,
    }

    multiclass_pred = np.argmax(lgb_A * lgb_pred + xgb_B * xgb_pred + tab_C * tab_pred + llm_D * llm_pred, axis=1)
    multiclass_pred_proba = lgb_A * lgb_pred + xgb_B * xgb_pred + tab_C * tab_pred + + llm_D * llm_pred
    
    # Feature importance
    fi_df = pd.DataFrame({'feature': X.columns, 'importance': lgb_model.feature_importances_})
    top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
    feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
    print(f"[S1] {feat_str}")
    
    # 예측 저장
    submission_final['S1'] = multiclass_pred
    for col in targets_binary:
      submission_final[col] = binary_preds[col]
    submission_final = submission_final[['subject_id', 'sleep_date', 'lifelog_date', 'Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']]
    fname = f"submission_{np.mean(total_avg_f1s)}.csv"
    submission_final.to_csv(fname, index=False)
    print(f"# {fname} 저장 완료")
    print(f"# submission shape:{submission_final.shape}")
    print("================================================")
    
    # ---------------------------------------- all Weight Combinations ----------------------------------------
    
    submission_final_dict = {}
    print("\nAll Weight Combinations:")
    for i, (weights, score) in enumerate(zip(all_weights, all_scores)):
    
        print(f"Rank {i+1}: lgb_A={weights[0]:.1f}, xgb_B={weights[1]:.1f}, tab_C={weights[2]:.1f}, llm_D={weights[3]:.1f} - Score: {score:.4f}")
        
        # Generate submission with these weights
        lgb_A, xgb_B, tab_C, llm_D = weights
        
        # Binary predictions
        for col in targets_binary:
            preds = binary_test_preds[col]
            ensemble_pred = (lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + 
                          tab_C * preds['tab'] + llm_D * preds['llm'] > 0.5).astype(int)
            submission_final[col] = ensemble_pred
        
        # Multiclass prediction
        preds = multiclass_test_preds
        ensemble_pred = np.argmax(lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + 
                               tab_C * preds['tab'] + llm_D * preds['llm'], axis=1)
        submission_final['S1'] = ensemble_pred
        
        fname = f"submission_top{i+1}_{score:.4f}.csv"
        submission_final.to_csv(fname, index=False)
        print(f"Saved submission to {fname}")

    # Use the best weights for final submission
    best_weights = all_weights[0]
    lgb_A, xgb_B, cat_C, llm_D = best_weights
    
    
    
    # 모델별 예측결과 비율 비교
    a11 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].sum()
    a13 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].apply(len)
    a12 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].mean()
    a21 = submission_final_dict[0][['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].sum()
    a23 = submission_final_dict[0][['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].apply(len)
    a22 = submission_final_dict[0][['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].mean()
    result = pd.concat([a11, a13, a12, a21, a23, a22], axis=1)
    result.columns = ['학습sum','학습len','학습mean','테스트sum','테스트len','테스트mean']
    print('\n STEP3: 예측결과 비교표')
    display(result)
    
    oof_result = []
    
    return submission_final_dict[0], oof_result

### ============================

### 📦 모델 학습

In [25]:
%%time

# 공통 하이퍼파라미터
common_params = {
  'n_estimators': 5000,
  "learning_rate": 0.01,
  # 'min_data_in_leaf':2,
  # 'bagging_fraction':0.9,
  # 'feature_fraction':0.6,
  'lambda_l1': 5,
  'lambda_l2': 1,
  # 'max_depth': 4,
  'n_jobs': -1,
  'verbosity': -1
}

# 모델별 세부 하이퍼파라미터
best_param_dict = {}
best_param_dict['Q3'] = common_params
best_param_dict['S1'] = common_params
best_param_dict['S2'] = common_params
best_param_dict['S3'] = common_params
best_param_dict['Q1'] = common_params
best_param_dict['Q2'] = common_params

submission_final, oof_result = run_basemodel(train, test, valid_ids, best_param_dict, n_splits=5, random_state=41)

light_week_type_lag1
weekday
week_type
week_type_lag1
activehour_top_bssid
beforebed_top_bssid
# X shape: (450, 262)
# test_X shape: (250, 262)

 STEP1: 실험 결과 확인
=============== Validation Results ==============


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 40/40 [00:07<00:00,  5.71it/s]


Q1 Epoch: 1: t_loss:0.257 t_acc:0.563 v_loss:0.207 v_acc:0.286


...

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 250/250 [00:42<00:00,  5.90it/s]


[S3] S3_te(623), light_night_mean(274), free_hour_rssi_mean(219), S3_te2(202), beforebed_메신저_time(199), light_wake_time_diff(173), light_rolling_wake_time_2d(151), sleep_time_lag2(143), 불끈시간부터기상시간(125), light_day_mean(125)


Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 250/250 [00:45<00:00,  5.50it/s]


[S1] S1_te2(691), light_sleep_time_ratio(641), m_activity_met@240min@sum@04h00m(538), wake_time_diff(520), S1_te(502), weekday_avg_sleep(425), light_wake_time_lag2(366), m_activity@240min@std@12h00m(356), sleep_duration_ratio(348), mgps_first_wakeup_minutes(337)
# submission_0.6469612085235824.csv 저장 완료
# submission shape:(250, 9)

All Weight Combinations:
Rank 1: lgb_A=0.2, xgb_B=0.5, tab_C=0.2, llm_D=0.1 - Score: 0.6667
Saved submission to submission_top1_0.6667.csv
Rank 2: lgb_A=0.3, xgb_B=0.5, tab_C=0.2, llm_D=0.0 - Score: 0.6667
Saved submission to submission_top2_0.6667.csv
Rank 3: lgb_A=0.4, xgb_B=0.3, tab_C=0.2, llm_D=0.1 - Score: 0.6667
Saved submission to submission_top3_0.6667.csv
Rank 4: lgb_A=0.3, xgb_B=0.4, tab_C=0.3, llm_D=0.0 - Score: 0.6667
Saved submission to submission_top4_0.6667.csv
Rank 5: lgb_A=0.4, xgb_B=0.3, tab_C=0.3, llm_D=0.0 - Score: 0.6667
Saved submission to submission_top5_0.6667.csv
Rank 6: lgb_A=0.4, xgb_B=0.4, tab_C=0.2, llm_D=0.0 - Score: 0.6667
Save

NameError: name 'top_3_weights' is not defined

### 📦 이전제출과 비교

In [27]:
from pathlib import Path

# Reference file
reference_file = '/content/drive/MyDrive/data/ch2025_data_items/share/submissions/'
reference_path = 'submission_top1_0.6492.csv'

# reference_path = '../input/'
# reference_file = "submission_top1_0.6492_best_top1.csv"

ref_df = pd.read_csv(reference_path + reference_file)

# Get all CSV files in data directory
data_dir = Path('./')
csv_files = list(data_dir.glob('*.csv'))

# Store differences for each file
differences = []

for csv_file in csv_files:
    if csv_file.name == os.path.basename(reference_file):
        continue

    # Read current file
    current_df = pd.read_csv(csv_file)

    # Calculate differences in specified columns
    diff_count = 0
    for col in ['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']:
        diff_count += (ref_df[col] != current_df[col]).sum()

    differences.append((csv_file.name, diff_count))
    # print(f"File: {csv_file.name}, Differences: {diff_count}")

# Sort by difference count and get top 20
differences.sort(key=lambda x: x[1])
print("\nTop 20 files with smallest differences:")
for i, (file_name, diff_count) in enumerate(differences[:20], 1):
    print(f"{str(i).zfill(2)}. {file_name}: {diff_count} differences")


Top 20 files with smallest differences:
01. submission_top41_0.6562.csv: 94 differences
02. submission_top4_0.6667.csv: 95 differences
03. submission_top88_0.6468.csv: 97 differences
04. submission_top1_0.6667.csv: 99 differences
05. submission_top2_0.6667.csv: 99 differences
06. submission_top58_0.6514.csv: 99 differences
07. submission_top40_0.6562.csv: 99 differences
08. submission_top22_0.6601.csv: 100 differences
09. submission_top42_0.6562.csv: 100 differences
10. submission_top35_0.6579.csv: 101 differences
11. submission_top73_0.6490.csv: 101 differences
12. submission_top48_0.6547.csv: 101 differences
13. submission_top28_0.6596.csv: 101 differences
14. submission_top13_0.6629.csv: 101 differences
15. submission_top100_0.6444.csv: 101 differences
16. submission_top159_0.6365.csv: 102 differences
17. submission_top97_0.6444.csv: 102 differences
18. submission_top67_0.6501.csv: 102 differences
19. submission_top24_0.6601.csv: 102 differences
20. submission_top6_0.6667.csv: 102 